# Лабораторная работа №1: Библиотека конструктивных вещественных чисел и алгоритмы оптимизации

**Команда (ФИО):**
1. Шайкин Андрей Сергеевич
2. Коханенко Вадим Вячеславович
3. Вяхирев Иван Олегович


In [ ]:
import time
import sys
import math
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from decimal import Decimal, getcontext
import typing as t

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Увеличиваем лимит рекурсии для глубоких конструктивных цепочек
sys.setrecursionlimit(50000)
# Системная точность (с запасом)
getcontext().prec = 200

# 1. КОНТЕКСТ И НАСТРОЙКИ (User Config)


In [ ]:
@dataclass
class ConstructiveContext:
    """
    Настройки ядра.
    max_deep_precision: "K" для внутренних сравнений (если разница < 10^-K, числа равны).
    display_precision: Сколько знаков показывать в print() и на графиках.
    """
    max_deep_precision: int = 50
    display_precision: int = 10

CTX = ConstructiveContext()

# 2. ЯДРО (CONSTRUCTIVE REALS)

In [ ]:
class ConstructiveReal:
    """
    Число, представленное алгоритмом вычисления.
    """
    def __init__(self, evaluator: t.Callable[[int], Decimal]):
        self._evaluator = evaluator
        self._cache: t.Dict[int, Decimal] = {}
        self._magnitude: t.Optional[int] = None

    def evaluate(self, k: int) -> Decimal:
        """Вернуть значение с точностью 10^-k"""
        if k in self._cache: return self._cache[k]
        # Берем запас точности +10 знаков, чтобы не накапливать ошибку округления
        getcontext().prec = k + 10
        val = self._evaluator(k)
        self._cache[k] = val
        return val

    # def consolidate(self) -> 'ConstructiveReal':
    #     """
    #     Сбрасывает историю вычислений
    #     Нужно вызывать периодически, иначе память переполнится историей операций.
    #     """
    #     # Вычисляем с высокой точностью
    #     val = self.evaluate(CTX.max_deep_precision + 5)
    #     s_val = str(val)
    #     # Возвращаем "чистое" число без хвоста вычислений
    #     return ConstructiveReal(lambda k: Decimal(s_val))
    # В нашей реализации не требуется, но на деле у нас есть физические ограничения, которые заставят нас сбрасывать историю
    # Расскомитить тут и в алгоритмах

    def _get_magnitude(self) -> int:
        """Оценка порядка числа для корректного умножения"""
        if self._magnitude is None:
            val = abs(self.evaluate(0))
            self._magnitude = 0 if val == 0 else len(str(int(val))) + 1
        return self._magnitude

    @classmethod
    def from_float(cls, value: float) -> 'ConstructiveReal':
        return cls(lambda k: Decimal(str(value)))

    # --- Арифметика с учетом погрешностей ---
    def __add__(self, other):
        other = self._ensure(other)
        return ConstructiveReal(lambda k: self.evaluate(k + 1) + other.evaluate(k + 1))

    def __sub__(self, other):
        other = self._ensure(other)
        return ConstructiveReal(lambda k: self.evaluate(k + 1) - other.evaluate(k + 1))

    def __mul__(self, other):
        other = self._ensure(other)
        # При умножении ошибка растет пропорционально значению множителя
        def mult(k):
            p_self = k + other._get_magnitude() + 2
            p_other = k + self._get_magnitude() + 2
            return self.evaluate(p_self) * other.evaluate(p_other)
        return ConstructiveReal(mult)

    def __truediv__(self, other):
        other = self._ensure(other)
        def div(k):
            # При делении точность зависит от того, насколько мал делитель
            extra = max(0, -other._get_magnitude()) + 5
            return self.evaluate(k + extra) / other.evaluate(k + extra)
        return ConstructiveReal(div)

    # --- Сравнение (динамическое довычисление) ---
    def __lt__(self, other):
        other = self._ensure(other)
        k = 2
        while k <= CTX.max_deep_precision:
            diff = self.evaluate(k) - other.evaluate(k)
            epsilon = Decimal(10) ** (-k + 1)
            if diff < -epsilon: return True
            if diff > epsilon: return False
            k *= 2
        return False

    def __gt__(self, other): return self._ensure(other).__lt__(self)
    def __le__(self, other): return self.__lt__(other) or not (self > other)
    def __ge__(self, other): return self.__gt__(other) or not (self < other)

    def _ensure(self, val):
        return val if isinstance(val, ConstructiveReal) else ConstructiveReal.from_float(val)

    def to_float(self) -> float:
        """Возвращает float для графиков и простых принтов"""
        return float(self.evaluate(CTX.display_precision))

    def __repr__(self):
        # Используем настройку пользователя для строкового представления
        val = self.to_float()
        return f"{val:.{CTX.display_precision}f}"

# 3. АБСТРАКЦИИ ОПТИМИЗАЦИИ И ФУНКЦИЙ

In [ ]:
@dataclass
class StopCriteria:
    tol_x: float = 1e-5
    tol_f: float = 1e-5
    max_iter: int = 1000
    max_calls: int = 5000

@dataclass
class OptimizationResult:
    best_x: t.List[float]
    best_f: float
    history_x: t.List[t.List[float]]
    history_f: t.List[float]
    iterations: int
    calls: int
    time_sec: float

class ObjectiveFunction(ABC):
    def __init__(self):
        self.calls_count = 0

    def __call__(self, x: t.List[ConstructiveReal]) -> ConstructiveReal:
        """Вычисление в конструктивных числах (для алгоритма)"""
        self.calls_count += 1
        return self._compute_constructive(x)

    @abstractmethod
    def _compute_constructive(self, x: t.List[ConstructiveReal]) -> ConstructiveReal:
        pass

    @abstractmethod
    def evaluate_numpy(self, X: np.ndarray, Y: np.ndarray) -> np.ndarray:
        """
        Вычисление в numpy (float) для отрисовки 3D поверхности.
        Это позволяет Визуализатору не знать формулу.
        """
        pass

    def reset(self):
        self.calls_count = 0

class Optimizer(ABC):
    @abstractmethod
    def optimize(self, func: ObjectiveFunction, x0: t.List[float], criteria: StopCriteria) -> OptimizationResult:
        pass

# 4. РЕАЛИЗАЦИЯ ЗАДАЧИ

In [ ]:
class Rosenbrock(ObjectiveFunction):
    """
    Целевая функция Розенброка.
    Реализует и конструктивную логику (для точности), и numpy логику (для графики).
    """
    def __init__(self, a=1.0, b=100.0):
        super().__init__()
        self.a_val = a
        self.b_val = b
        self.a = ConstructiveReal.from_float(a)
        self.b = ConstructiveReal.from_float(b)

    def _compute_constructive(self, x: t.List[ConstructiveReal]) -> ConstructiveReal:
        res = ConstructiveReal.from_float(0.0)
        for i in range(len(x) - 1):
            term1 = self.b * (x[i+1] - (x[i] * x[i])) * (x[i+1] - (x[i] * x[i]))
            term2 = (self.a - x[i]) * (self.a - x[i])
            res = res + term1 + term2
        return res

    def evaluate_numpy(self, X: np.ndarray, Y: np.ndarray) -> np.ndarray:
        # Формула для визуализации (только 2D срез)
        # Z = (a - x)^2 + b * (y - x^2)^2
        return (self.a_val - X)**2 + self.b_val * (Y - X**2)**2

# 5. АЛГОРИТМЫ ОПТИМИЗАЦИИ

In [ ]:
class CoordinateDescent(Optimizer):
    def __init__(self, step=0.5, decay=2.0):
        self.step, self.decay = step, decay

    def optimize(self, func, x0, crit):
        start_t = time.time()
        func.reset()

        x = [ConstructiveReal.from_float(v) for v in x0]
        step = ConstructiveReal.from_float(self.step)
        decay_cr = ConstructiveReal.from_float(self.decay)
        c_tol_x = ConstructiveReal.from_float(crit.tol_x)
        c_tol_f = ConstructiveReal.from_float(crit.tol_f)

        hist_x, hist_f = [[v.to_float() for v in x]], [func(x).to_float()]
        f_curr = func(x)

        for i in range(crit.max_iter):

            if func.calls_count >= crit.max_calls:
              break

            # Оптимизация памяти (сброс истории каждые 15 итераций)
            # if i % 15 == 0:
            #     x = [v.consolidate() for v in x]
            #     step = step.consolidate()
            #     f_curr = f_curr.consolidate()

            moved = False
            f_prev = f_curr

            for dim in range(len(x)):
                saved_x = x[dim]

                # Вперед
                x[dim] = saved_x + step
                f_new = func(x)

                if func.calls_count >= crit.max_calls:
                    if f_new < f_curr: f_curr = f_new # Сохраняем последнее
                    break

                if f_new < f_curr:
                    f_curr, moved = f_new, True
                    continue

                # Назад
                x[dim] = saved_x - step
                f_new = func(x)

                if func.calls_count >= crit.max_calls:
                    if f_new < f_curr: f_curr = f_new
                    break

                if f_new < f_curr:
                    f_curr, moved = f_new, True
                    continue

                x[dim] = saved_x

            if func.calls_count >= crit.max_calls:
                hist_x.append([v.to_float() for v in x])
                hist_f.append(f_curr.to_float())
                break

            hist_x.append([v.to_float() for v in x])
            hist_f.append(f_curr.to_float())

            if not moved:
                step = step / decay_cr
                if step < c_tol_x: break
            elif (f_prev - f_curr) < c_tol_f and step < c_tol_x:
                break

        return OptimizationResult(
            [v.to_float() for v in x], f_curr.to_float(), hist_x, hist_f, i+1, func.calls_count, time.time()-start_t
        )

class NelderMead(Optimizer):
    def __init__(self, step=0.5):
        self.step = step

    def optimize(self, func, x0, crit):
        start_t = time.time()
        func.reset()
        n = len(x0)
        c_tol_f = ConstructiveReal.from_float(crit.tol_f)

        # Симплекс
        base = [ConstructiveReal.from_float(v) for v in x0]
        simplex = [(base, func(base))]
        step_cr = ConstructiveReal.from_float(self.step)

        for i in range(n):
            pt = base.copy()
            pt[i] = pt[i] + step_cr
            simplex.append((pt, func(pt)))

        hist_x, hist_f = [[v.to_float() for v in base]], [simplex[0][1].to_float()]

        # Параметры метода
        alpha = ConstructiveReal.from_float(1.0)
        gamma = ConstructiveReal.from_float(2.0)
        rho = ConstructiveReal.from_float(0.5)
        sigma = ConstructiveReal.from_float(0.5)

        for i in range(crit.max_iter):
            if func.calls_count >= crit.max_calls:
                break

            # Оптимизация памяти (сброс истории каждые 15 итераций)
            # if i % 15 == 0:
            #     clean_simplex = []
            #     for pt, val in simplex:
            #         clean_simplex.append(([p.consolidate() for p in pt], val.consolidate()))
            #     simplex = clean_simplex

            simplex.sort(key=lambda item: item[1])
            best, good, worst = simplex[0], simplex[-2], simplex[-1]

            hist_x.append([v.to_float() for v in best[0]])
            hist_f.append(best[1].to_float())

            if worst[1] - best[1] < c_tol_f:
                break

            # Центроид
            mid = []
            n_cr = ConstructiveReal.from_float(n)
            for d in range(n):
                s = sum((simplex[k][0][d] for k in range(n)), ConstructiveReal.from_float(0.0))
                mid.append(s / n_cr)

            # Отражение
            xr = [mid[d] + alpha * (mid[d] - worst[0][d]) for d in range(n)]
            fxr = func(xr)

            if best[1] <= fxr < good[1]:
                simplex[-1] = (xr, fxr)
                continue

            # Расширение
            if fxr < best[1]:
                xe = [mid[d] + gamma * (xr[d] - mid[d]) for d in range(n)]
                fxe = func(xe)
                simplex[-1] = (xe, fxe) if fxe < fxr else (xr, fxr)
                continue

            # Сжатие
            xc = [mid[d] + rho * (worst[0][d] - mid[d]) for d in range(n)]
            fxc = func(xc)
            if fxc < worst[1]:
                simplex[-1] = (xc, fxc)
                continue

            # Глобальное сжатие
            for k in range(1, n + 1):
                pt = [best[0][d] + sigma * (simplex[k][0][d] - best[0][d]) for d in range(n)]
                simplex[k] = (pt, func(pt))

        return OptimizationResult(
            [v.to_float() for v in simplex[0][0]], simplex[0][1].to_float(), hist_x, hist_f, i+1, func.calls_count, time.time()-start_t
        )

# 6. ВИЗУАЛИЗАЦИЯ

In [ ]:
class Visualizer:
    @staticmethod
    def show(func: ObjectiveFunction, results: t.Dict[str, OptimizationResult]):
        """
        Рисует результаты. Не знает формулу.
        """
        # Сетка для фона
        x_grid = np.linspace(-2.0, 2.5, 100)
        y_grid = np.linspace(-1.5, 3.0, 100)
        X, Y = np.meshgrid(x_grid, y_grid)

        # Мы спрашиваем у функции её значения numpy-методом
        Z = func.evaluate_numpy(X, Y)

        fig = make_subplots(
            rows=1, cols=2,
            specs=[[{'type': 'surface'}, {'type': 'xy'}]],
            subplot_titles=('3D Ландшафт', 'Сходимость (Log Scale)')
        )

        # 3D Поверхность
        fig.add_trace(go.Surface(z=Z, x=X, y=Y, colorscale='Viridis', opacity=0.7, showscale=False), row=1, col=1)

        colors = ['cyan', 'red', 'lime', 'yellow']

        for idx, (name, res) in enumerate(results.items()):
            c = colors[idx % len(colors)]
            hx = np.array(res.history_x)

            # Траектория
            fig.add_trace(go.Scatter3d(
                x=hx[:, 0], y=hx[:, 1], z=res.history_f,
                mode='lines+markers', marker=dict(size=3, color=c),
                line=dict(color=c, width=4), name=name
            ), row=1, col=1)

            # График сходимости
            fig.add_trace(go.Scatter(
                y=res.history_f, mode='lines',
                line=dict(color=c, width=2), name=name
            ), row=1, col=2)

        fig.update_yaxes(type="log", title_text="f(x)", row=1, col=2)
        fig.update_xaxes(title_text="Итерации", row=1, col=2)
        fig.update_layout(height=600, width=1200, template='plotly_dark', margin=dict(l=10, r=10, b=10, t=40))
        fig.show()

# 7. MAIN (Точка входа)

In [ ]:
if __name__ == "__main__":

    # 1. Настройка ядра (K и точность вывода)
    CTX.max_deep_precision = 50   # Глубина сравнения (точность вычислений)
    CTX.display_precision = 10

    # 2. Выбор функции (Можешь заменить Rosenbrock на другую реализацию ObjectiveFunction)
    target_func = Rosenbrock(a=1.0, b=100.0)

    # 3. Критерии останова
    criteria = StopCriteria(max_iter=4000, max_calls = 8000, tol_x=1e-5, tol_f=1e-6)
    start_point = [-1.5, 2.0]

    # 4. Стратегии оптимизации
    strategies = {
        "Coordinate Descent": CoordinateDescent(),
        "Nelder-Mead": NelderMead()
    }

    print(f"--- ЗАПУСК (Display Precision: {CTX.display_precision}) ---")
    results = {}

    for name, algo in strategies.items():
        print(f"Выполняется: {name}...")
        res = algo.optimize(target_func, start_point, criteria)
        results[name] = res

        # Динамическое форматирование строки на основе CTX.display_precision
        fmt = f".{CTX.display_precision}f"

        print(f"  > Минимум: {[float(format(x, fmt)) for x in res.best_x]}")
        print(f"  > Значение f(x): {format(res.best_f, fmt)}")
        print(f"  > Итераций: {res.iterations}, Вызовы f(x): {res.calls}, Время: {res.time_sec:.4f}s\n")

    # Передаем саму функцию, чтобы визуализатор знал, какую поверхность рисовать
    Visualizer.show(target_func, results)

--- ЗАПУСК (Display Precision: 10) ---
Выполняется: Coordinate Descent...
  > Минимум: [0.9891662598, 0.9784851074]
  > Значение f(x): 0.0001167729
  > Итераций: 3010, Вызовы f(x): 8000, Время: 23.1167s

Выполняется: Nelder-Mead...
  > Минимум: [1.0003212618, 1.0006908503]
  > Значение f(x): 0.0000003358
  > Итераций: 99, Вызовы f(x): 178, Время: 3.1421s



### Вывод

**1. Выбор алгоритмов:**
Для работы с конструктивными числами (КВЧ) были выбраны методы нулевого порядка (без производных): **Покоординатный спуск** и **Нелдер-Мид**.
*Причина:* В КВЧ любое сравнение требует вычислений с некоторой точностью. Градиентные методы требуют вычисления разности $f(x+h) - f(x)$ при $h \to 0$. Это приводит к вычитанию очень близких чисел, что заставляет алгоритм КВЧ бесконечно увеличивать точность (depth) для поиска различий, вызывая переполнение памяти и времени. Методы, основанные только на сравнении значений (`<`, `>`), лишены этого недостатка.

**2. Анализ результатов:**
*   **Покоординатный спуск:** Показал характерную "ступенчатую" сходимость. На изогнутом дне оврага Розенброка алгоритм вынужден бесконечно дробить шаг, так как направления осей не совпадают с направлением оврага. Это привело к медленной сходимости.
*   **Нелдер-Мид:** Показал превосходство адаптивной метрики. Симплекс автоматически деформировался ("вытянулся") вдоль дна оврага, что позволило алгоритму быстро достичь глобального минимума `(1, 1)` с высокой точностью.


#ДОП: UNIT-ТЕСТЫ: ПОКРЫТИЕ ЯДРА И АЛГОРИТМОВ

In [ ]:
import unittest
from decimal import Decimal

class TestConstructiveReals(unittest.TestCase):
    def setUp(self):
        # Перед каждым тестом сбрасываем настройки ядра
        CTX.max_deep_precision = 50
        CTX.display_precision = 10

    def test_basic_arithmetic(self):
        """Проверка базовой арифметики и проблемы 0.1 + 0.2"""
        a = ConstructiveReal.from_float(0.1)
        b = ConstructiveReal.from_float(0.2)
        c = a + b

        # В классическом float 0.1 + 0.2 = 0.30000000000000004
        # В конструктивных числах разница с 0.3 должна быть строго меньше 10^-10
        diff = abs(c.evaluate(10) - Decimal('0.3'))
        self.assertTrue(diff < Decimal('1e-10'), "Ошибка сложения конструктивных чисел")

    def test_multiplication_precision_scaling(self):
        """Проверка честного умножения (распространение ошибки)"""
        # Умножаем большое число на маленькое.
        # Если бы мы не запрашивали повышенную точность для операндов, ошибка бы выросла.
        a = ConstructiveReal.from_float(1000000.0)
        b = ConstructiveReal.from_float(0.000001)
        c = a * b

        # Ожидаем 1.0
        val = c.evaluate(10)
        self.assertTrue(abs(val - Decimal('1.0')) < Decimal('1e-10'), "Ошибка масштабирования точности при умножении")

    def test_dynamic_comparison_and_lazy_evaluation(self):
        """
        САМЫЙ ВАЖНЫЙ ТЕСТ (Специфика конструктивной математики).
        Проверяем, как алгоритм "довычисляет" разницу между близкими числами.
        """
        # a = 1.0, b = 1.0 + 10^-15
        a = ConstructiveReal(lambda k: Decimal('1.0'))
        b = ConstructiveReal(lambda k: Decimal('1.0') + Decimal('1e-15'))

        # 1. Ограничиваем глубину поиска точностью 10 знаков
        CTX.max_deep_precision = 10
        # Алгоритм проверит k=2, 4, 8. Дальше нельзя. Разницу не увидит.
        self.assertFalse(a < b, "При низкой точности числа должны считаться равными")

        # 2. Увеличиваем лимит глубины поиска до 32 знаков
        CTX.max_deep_precision = 32
        # Теперь алгоритм сделает шаги k=16 и k=32. На k=32 он 100% увидит разницу.
        self.assertTrue(a < b, "При высокой точности алгоритм должен довычислить разницу")


class TestObjectiveFunctions(unittest.TestCase):
    def test_rosenbrock_minimum(self):
        """Проверка функции Розенброка в точке глобального минимума (1, 1)"""
        rosenbrock = Rosenbrock(a=1.0, b=100.0)
        x =[ConstructiveReal.from_float(1.0), ConstructiveReal.from_float(1.0)]

        result = rosenbrock(x)
        # В точке (1, 1) функция должна быть строго 0
        self.assertTrue(abs(result.evaluate(5) - Decimal('0.0')) < Decimal('1e-5'))

    def test_rosenbrock_point(self):
        """Проверка значения функции Розенброка в (0, 0)"""
        rosenbrock = Rosenbrock(a=1.0, b=100.0)
        x =[ConstructiveReal.from_float(0.0), ConstructiveReal.from_float(0.0)]

        result = rosenbrock(x)
        # В точке (0, 0) f(x) = (1-0)^2 + 100*(0-0)^2 = 1.0
        self.assertTrue(abs(result.evaluate(5) - Decimal('1.0')) < Decimal('1e-5'))


class TestOptimizationAlgorithms(unittest.TestCase):
    def setUp(self):
        self.func = Rosenbrock()
        self.x0 =[-1.0, 1.0]
        # Ставим маленькие лимиты, чтобы тесты проходили мгновенно
        # и не вызывали RecursionError (т.к. мы убрали consolidate)
        self.criteria = StopCriteria(max_iter=15, max_calls=100, tol_x=1e-5, tol_f=1e-5)

    def test_coordinate_descent_decreases_function(self):
        """Проверка, что Покоординатный спуск реально минимизирует функцию"""
        f_start = self.func([ConstructiveReal.from_float(v) for v in self.x0]).to_float()

        optimizer = CoordinateDescent(step=0.2)
        res = optimizer.optimize(self.func, self.x0, self.criteria)

        # После оптимизации значение функции должно стать меньше стартового
        self.assertTrue(res.best_f < f_start, "Покоординатный спуск не уменьшил значение функции")

    def test_nelder_mead_decreases_function(self):
        """Проверка, что алгоритм Нелдера-Мида реально минимизирует функцию"""
        f_start = self.func([ConstructiveReal.from_float(v) for v in self.x0]).to_float()

        optimizer = NelderMead(step=0.2)
        res = optimizer.optimize(self.func, self.x0, self.criteria)

        # После оптимизации значение функции должно стать меньше стартового
        self.assertTrue(res.best_f < f_start, "Нелдер-Мид не уменьшил значение функции")
        # Убедимся, что история сохраняется
        self.assertTrue(len(res.history_f) > 0)


if __name__ == '__main__':

    unittest.main(argv=['first-arg-is-ignored'], exit=False, verbosity=2)

test_basic_arithmetic (__main__.TestConstructiveReals.test_basic_arithmetic)
Проверка базовой арифметики и проблемы 0.1 + 0.2 ... ok
test_dynamic_comparison_and_lazy_evaluation (__main__.TestConstructiveReals.test_dynamic_comparison_and_lazy_evaluation)
САМЫЙ ВАЖНЫЙ ТЕСТ (Специфика конструктивной математики). ... ok
test_multiplication_precision_scaling (__main__.TestConstructiveReals.test_multiplication_precision_scaling)
Проверка честного умножения (распространение ошибки) ... ok
test_rosenbrock_minimum (__main__.TestObjectiveFunctions.test_rosenbrock_minimum)
Проверка функции Розенброка в точке глобального минимума (1, 1) ... ok
test_rosenbrock_point (__main__.TestObjectiveFunctions.test_rosenbrock_point)
Проверка значения функции Розенброка в (0, 0) ... ok
test_coordinate_descent_decreases_function (__main__.TestOptimizationAlgorithms.test_coordinate_descent_decreases_function)
Проверка, что Покоординатный спуск реально минимизирует функцию ... ok
test_nelder_mead_decreases_functio